In [1]:
!pip install transformers datasets peft accelerate bitsandbytes trl torch -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 751.0/751.0 kB 65.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 47.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 13.4 MB/s eta 0:00:00


In [2]:
from google.colab import files

uploaded = files.upload()  # upload jenkins_errors.jsonl here
print("File uploaded successfully!")


Saving jenkins_errors.jsonl to jenkins_errors.jsonl
File uploaded successfully!


In [3]:
import json

# Load the dataset
data = []
with open('jenkins_errors.jsonl', 'r') as f:
    for line in f:
        data.append(json.loads(line.strip()))

print(f"Total entries: {len(data)}")
print("\nSample entry:")
print(json.dumps(data[0], indent=2))

Total entries: 201

Sample entry:
{
  "instruction": "You are a Jenkins DevOps expert. Analyze this Jenkins pipeline error and give: 1) What went wrong 2) Root cause 3) How to fix it. Be short and clear.",
  "input": "ERROR: java.lang.NullPointerException at com.app.service.UserService.getUser(UserService.java:42) at com.app.controller.UserController.fetch(UserController.java:18) FATAL: Test suite failed ERROR: script returned exit code 1",
  "output": "1) What went wrong: A unit test crashed with a NullPointerException. 2) Root cause: An object at UserService.java line 42 was used before being initialized. 3) Fix: Add a null check before accessing the object at line 42, or ensure the object is properly initialized in the constructor or test setup method."
}


In [4]:
from datasets import Dataset

def format_prompt(example):
    return {
        "text": f"""### Instruction:
{example['instruction']}

### Input:
{example['input']}

### Response:
{example['output']}"""
    }

# Convert to HuggingFace dataset
dataset = Dataset.from_list(data)
dataset = dataset.map(format_prompt)

print(f"Dataset size: {len(dataset)}")
print("\nFormatted sample:")
print(dataset[0]['text'])

Map:   0%|          | 0/201 [00:00<?, ? examples/s]

Dataset size: 201

Formatted sample:
### Instruction:
You are a Jenkins DevOps expert. Analyze this Jenkins pipeline error and give: 1) What went wrong 2) Root cause 3) How to fix it. Be short and clear.

### Input:
ERROR: java.lang.NullPointerException at com.app.service.UserService.getUser(UserService.java:42) at com.app.controller.UserController.fetch(UserController.java:18) FATAL: Test suite failed ERROR: script returned exit code 1

### Response:
1) What went wrong: A unit test crashed with a NullPointerException. 2) Root cause: An object at UserService.java line 42 was used before being initialized. 3) Fix: Add a null check before accessing the object at line 42, or ensure the object is properly initialized in the constructor or test setup method.


In [5]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# 4-bit quantization config — saves memory
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

print("Model loaded successfully!")
print(f"Model parameters: {model.num_parameters():,}")

Loading tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Loading model...


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model loaded successfully!
Model parameters: 1,100,048,384


In [6]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Prepare model for training
model = prepare_model_for_kbit_training(model)

# LoRA configuration
lora_config = LoraConfig(
    r=16,                          # LoRA rank
    lora_alpha=32,                 # LoRA scaling
    target_modules=[               # Which layers to apply LoRA to
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# Apply LoRA to model
model = get_peft_model(model, lora_config)

# Print trainable parameters
model.print_trainable_parameters()

trainable params: 12,615,680 || all params: 1,112,664,064 || trainable%: 1.1338


In [7]:
def tokenize(example):
    result = tokenizer(
        example["text"],
        truncation=True,
        max_length=512,
        padding="max_length",
    )
    result["labels"] = result["input_ids"].copy()
    return result

print("Tokenizing dataset...")
tokenized_dataset = dataset.map(tokenize, remove_columns=dataset.column_names)

# Split into train and eval
split = tokenized_dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split["train"]
eval_dataset  = split["test"]

print(f"Train samples : {len(train_dataset)}")
print(f"Eval samples  : {len(eval_dataset)}")

Tokenizing dataset...


Map:   0%|          | 0/201 [00:00<?, ? examples/s]

Train samples : 180
Eval samples  : 21


In [15]:
from transformers import TrainingArguments
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="./jenkins-tinyllama",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    warmup_steps=10,
    learning_rate=2e-4,
    fp16=False,           # ✅ disabled
    bf16=True,            # ✅ use bfloat16 instead — matches model dtype
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none",
    optim="paged_adamw_8bit",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    args=training_args,
    formatting_func=lambda x: x["text"],
)

print("Trainer configured successfully!")

Trainer configured successfully!


In [16]:
print("Starting fine-tuning...")
print("This will take approximately 10-20 minutes on T4 GPU")
print("="*50)

trainer.train()

print("="*50)
print("Fine-tuning complete! ✅")

Starting fine-tuning...
This will take approximately 10-20 minutes on T4 GPU


Epoch,Training Loss,Validation Loss
1,2.027238,0.714992
2,0.622224,0.458345
3,0.436020,0.438769


Fine-tuning complete! ✅


In [17]:
import os

print("Saving model...")
os.makedirs("./jenkins-tinyllama-adapter", exist_ok=True)

trainer.model.save_pretrained("./jenkins-tinyllama-adapter")
tokenizer.save_pretrained("./jenkins-tinyllama-adapter")

print("Saved! ✅")
print("Files:")
for f in os.listdir("./jenkins-tinyllama-adapter"):
    print(f"  - {f}")

Saving model...
Saved! ✅
Files:
  - tokenizer_config.json
  - adapter_model.safetensors
  - adapter_config.json
  - README.md
  - chat_template.jinja
  - tokenizer.json


In [18]:
import torch

model.eval()

test_prompt = """### Instruction:
You are a Jenkins DevOps expert. Analyze this Jenkins pipeline error and give: 1) What went wrong 2) Root cause 3) How to fix it. Be short and clear.

### Input:
ERROR: java.lang.NullPointerException at UserService.java:42 ERROR: Test suite failed exit code 1

### Response:"""

print("Testing model...")
print("="*50)

inputs = tokenizer(
    test_prompt,
    return_tensors="pt"
).to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
        repetition_penalty=1.2
    )

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("AI Response:")
print("="*50)
print(response.split("### Response:")[-1].strip())
print("="*50)


Testing model...
AI Response:



In [19]:
import torch

model.eval()

test_prompt = """### Instruction:
You are a Jenkins DevOps expert. Analyze this Jenkins pipeline error and give: 1) What went wrong 2) Root cause 3) How to fix it. Be short and clear.

### Input:
ERROR: java.lang.NullPointerException at UserService.java:42 ERROR: Test suite failed exit code 1

### Response:"""

print("Testing model...")
print("="*50)

inputs = tokenizer(
    test_prompt,
    return_tensors="pt",
    truncation=True,
    max_length=512
).to("cuda")

with torch.no_grad():
    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=200,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
        repetition_penalty=1.2,
        min_new_tokens=20        # ✅ force at least 20 tokens output
    )

# Decode only the NEW tokens (not the input prompt)
input_length = inputs["input_ids"].shape[1]
new_tokens = outputs[0][input_length:]   # ✅ slice off input
response = tokenizer.decode(new_tokens, skip_special_tokens=True)

print("AI Response:")
print("="*50)
print(response)
print("="*50)

Testing model...
AI Response:

1) What went wrong: The test execution passed but the test suite is failing because there was a NullPointerException in the user service method. 2) Root cause: A local variable of type Integer was initialized as null, causing an exception when calling the service method. 3) Fix: Verify that all variables used by the tests have been initialized before use or check for NPEs inside the method where it can be introduced during testing.


In [21]:
!pip install torchao --upgrade -q
print("torchao upgraded! ✅")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 39.8 MB/s eta 0:00:00
torchao upgraded! ✅


In [22]:
from peft import PeftModel
from transformers import AutoModelForCausalLM
import torch

print("Loading base model for merging...")
base_model = AutoModelForCausalLM.from_pretrained(
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    dtype=torch.float16,
    device_map="cpu"
)

print("Merging LoRA adapter into base model...")
merged_model = PeftModel.from_pretrained(
    base_model,
    "./jenkins-tinyllama-adapter"
)
merged_model = merged_model.merge_and_unload()
print("Merge complete! ✅")

print("Saving merged model...")
merged_model.save_pretrained(
    "./jenkins-tinyllama-merged",
    safe_serialization=True
)
tokenizer.save_pretrained("./jenkins-tinyllama-merged")

print("Merged model saved! ✅")
print("Files:")
import os
for f in os.listdir("./jenkins-tinyllama-merged"):
    print(f"  - {f}")

Loading base model for merging...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Merging LoRA adapter into base model...
Merge complete! ✅
Saving merged model...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Merged model saved! ✅
Files:
  - tokenizer_config.json
  - model.safetensors
  - generation_config.json
  - chat_template.jinja
  - config.json
  - tokenizer.json


In [26]:
import os

# Step 1 - Download the original tokenizer.model from HuggingFace
print("Downloading tokenizer.model...")
!wget -q https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0/resolve/main/tokenizer.model \
    -O ./jenkins-tinyllama-merged/tokenizer.model

# Verify it downloaded
size = os.path.getsize("./jenkins-tinyllama-merged/tokenizer.model") / 1024
print(f"tokenizer.model downloaded! Size: {size:.1f} KB ✅")

# Step 2 - Convert to GGUF
print("\nConverting to GGUF...")
!python llama.cpp/convert_hf_to_gguf.py \
    ./jenkins-tinyllama-merged \
    --outfile jenkins-expert.gguf \
    --outtype q8_0

# Step 3 - Verify
if os.path.exists("jenkins-expert.gguf"):
    size = os.path.getsize("jenkins-expert.gguf") / (1024*1024)
    print(f"\nGGUF created successfully! ✅")
    print(f"Size: {size:.1f} MB")
else:
    print("❌ Still failing, share error above")

tokenizer.model downloaded! Size: 488.0 KB ✅

Converting to GGUF...
INFO:hf-to-gguf:Loading model: jenkins-tinyllama-merged
INFO:hf-to-gguf:Model architecture: LlamaForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:output.weight,               torch.float16 --> Q8_0, shape = {2048, 32000}
INFO:hf-to-gguf:token_embd.weight,           torch.float16 --> Q8_0, shape = {2048, 32000}
INFO:hf-to-gguf:blk.0.attn_norm.weight,      torch.float16 --> F32, shape = {2048}
INFO:hf-to-gguf:blk.0.ffn_down.weight,       torch.float16 --> Q8_0, shape = {5632, 2048}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,       torch.float16 --> Q8_0, shape = {2048, 5632}
INFO:hf-to-gguf:blk.0.ffn_up.weight,         torch.float16 --> Q8_0, shape = {2048, 5632}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,       torch.float16 --> F32, shape = {2048}
INFO:hf-to-gguf:blk.0.attn_k.weight,   

In [27]:
modelfile_content = '''FROM ./jenkins-expert.gguf

SYSTEM """
You are an expert Jenkins and DevOps engineer with deep knowledge of:
- Jenkins pipeline errors and fixes
- Docker build failures
- Kubernetes deployment issues
- Database migration errors
- Python Java Node.js dependency errors
- Cloud provider AWS Azure GCP errors

When given a Jenkins error log always respond with:
1) What went wrong (1 sentence)
2) Root cause (1-2 sentences)
3) Exact fix with commands
Keep responses short and clear.
"""

PARAMETER temperature 0.7
PARAMETER top_p 0.9
PARAMETER repeat_penalty 1.2
'''

with open("Modelfile", "w") as f:
    f.write(modelfile_content)

print("Modelfile created! ✅")
print(modelfile_content)

Modelfile created! ✅
FROM ./jenkins-expert.gguf

SYSTEM """
You are an expert Jenkins and DevOps engineer with deep knowledge of:
- Jenkins pipeline errors and fixes
- Docker build failures
- Kubernetes deployment issues
- Database migration errors
- Python Java Node.js dependency errors
- Cloud provider AWS Azure GCP errors

When given a Jenkins error log always respond with:
1) What went wrong (1 sentence)
2) Root cause (1-2 sentences)
3) Exact fix with commands
Keep responses short and clear.
"""

PARAMETER temperature 0.7
PARAMETER top_p 0.9
PARAMETER repeat_penalty 1.2



In [28]:
from google.colab import files

print("Downloading jenkins-expert.gguf (~1.1GB)...")
print("This may take a few minutes...")
files.download("jenkins-expert.gguf")

print("Downloading Modelfile...")
files.download("Modelfile")

print("Both files downloading! ✅")
print("Check your browser downloads folder")

This may take a few minutes...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Both files downloading! ✅
Check your browser downloads folder
